In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a Github repository.

    Args:
    repo_owner: Github username or organization
    repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download the repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith ('.md')
                or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors = 'ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print("f\"Error processing {filename}: {e}\"")
            continue

    zf.close()
    return repository_data

In [2]:
# Calling the function for DuckDB
duckdb_docs = read_repo_data('duckdb', 'duckdb')

# Verify the results
print(f"Total documentation files ingested: {len(duckdb_docs)}")
print(f"Sample filename: {duckdb_docs[0]['filename']}")

Total documentation files ingested: 31
Sample filename: duckdb-main/.github/patches/extensions/README.md


In [3]:
print(duckdb_docs[0]) # Look at the first document's data

{'content': '# Extension patches\nPatches in this directory are used to smoothen the process of introducing changes to DuckDB that break compatibility with an\nout-of-tree extension. Extensions installed from git urls can automatically apply patches found in this directory. The APPLY_PATCHES flag \nshould be used to explicitly enable this feature. For example,\nlets say our extension config looks like this:\n\n```shell\nduckdb_extension_load(spatial\n    DONT_LINK\n    GIT_URL https://github.com/duckdb/duckdb_spatial\n    GIT_TAG f577b9441793f9170403e489f5d3587e023a945f\n    APPLY_PATCHES\n)\n```\nIn this example, upon downloading the spatial extension, all patches in the `.github/patches/extensions/spatial/*.patch`\nwill be automatically applied.\n\nNote that the reason for having the APPLY_PATCHES flag explicitly enabled is to make it easier for developers reading\nthe extension config to detect a patch is present. For this reason, the patching mechanism will actually fail if `APPLY_